<a href="https://colab.research.google.com/github/frank-morales2020/AST/blob/main/timesfm_3_0_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

https://research.google/blog/timesfm-3-a-zero-shot-foundation-model-for-multivariate-forecasting/?utm_source=linkedin&utm_medium=social&utm_campaign=social_post&utm_content=gr-acct

In [ ]:
!pip install timesfm[torch]

In [1]:
import numpy as np
from timesfm3 import TimesFM3Evaluator, ModelConfig

# 1. Initialize configuration and load TimesFM 3.0 from Hugging Face
config = ModelConfig(
    checkpoint_path="google/timesfm-3.0-pytorch",
    per_core_batch_size=16,
    device="cuda"  # Switch to "cpu" if running without a GPU
)
forecaster = TimesFM3Evaluator(config)

# 2. Prepare mock univariate and multivariate data
ts1 = np.linspace(0, 1, 100).astype(np.float32)
ts2 = np.sin(np.linspace(0, 24, 72)).astype(np.float32)

context_len = 128
horizon = 24
target_multivariate = np.random.randn(3, context_len).astype(np.float32)
past_only_cov = np.random.randn(1, context_len).astype(np.float32)
past_future_cov = np.random.randn(2, context_len + horizon).astype(np.float32)

# 3. Run predictions
univariate_outputs = list(forecaster.predict_batch([ts1, ts2], horizon=12, return_quantiles=True, use_symmetric_averaging=False))
multivariate_outputs = list(forecaster.predict_batch(contexts=[target_multivariate], horizon=horizon, past_only_covariates=[past_only_cov], past_future_covariates=[past_future_cov], return_quantiles=True, use_symmetric_averaging=False))

# 4. Display results
print("Univariate Series 1 Forecast Shape:", univariate_outputs[0].forecast.shape)
print("Univariate Series 1 Quantiles Shape:", univariate_outputs[0].quantiles.shape)
print("Multivariate Forecast Shape:", multivariate_outputs[0].forecast.shape)
print("Multivariate Quantiles Shape:", multivariate_outputs[0].quantiles.shape)

config.json:   0%|          | 0.00/1.27k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.32GB            

model.safetensors: downloading bytes:           |  0.00B            

Univariate Series 1 Forecast Shape: (12,)
Univariate Series 1 Quantiles Shape: (12, 9)
Multivariate Forecast Shape: (3, 24)
Multivariate Quantiles Shape: (3, 24, 9)


In [3]:
import numpy as np
from timesfm3 import TimesFM3Forecaster

# 1. Load the model from Hugging Face
forecaster = TimesFM3Forecaster.from_pretrained("google/timesfm-3.0-pytorch")

# 2. Prepare historical context data
context_data = np.sin(np.linspace(0, 30, 120)).astype(np.float32)

# 3. Run zero-shot inference
forecast_horizon = 32
output = forecaster.predict(
    context=context_data,
    horizon=forecast_horizon,
    return_quantiles=True
)

# 4. Extract point forecasts and confidence bounds
point_forecast = output.forecast
lower_bound = output.quantiles[:, 0]   # 10th percentile
upper_bound = output.quantiles[:, -1]  # 90th percentile

print("Point Forecast:", point_forecast)
print("Forecast Quantiles Shape:", output.quantiles.shape)

for i in range(5):
    print(f"Step {i+1}: Point = {point_forecast[i]:.3f} | 80% Range = [{lower_bound[i]:.3f}, {upper_bound[i]:.3f}]")

Point Forecast: [-0.91734254 -0.7926465  -0.6156823  -0.39753613 -0.15564924  0.09694236
  0.33751255  0.56216973  0.75281966  0.8926903   0.96960175  0.99210113
  0.9509687   0.8532924   0.7030693   0.5041832   0.27306244  0.02366358
 -0.22705771 -0.4652907  -0.671691   -0.8353816  -0.947692   -0.9969007
 -0.98386145 -0.9077907  -0.7816252  -0.59795594 -0.37837613 -0.13541126
  0.11119571  0.356791  ]
Forecast Quantiles Shape: (32, 9)
Step 1: Point = -0.917 | 80% Range = [-0.925, -0.911]
Step 2: Point = -0.793 | 80% Range = [-0.802, -0.787]
Step 3: Point = -0.616 | 80% Range = [-0.626, -0.607]
Step 4: Point = -0.398 | 80% Range = [-0.411, -0.387]
Step 5: Point = -0.156 | 80% Range = [-0.170, -0.141]


In [4]:
import numpy as np
from timesfm3 import TimesFM3Evaluator, ModelConfig

# Initialize production configuration for scaled batching and GPU acceleration
config = ModelConfig(
    checkpoint_path="google/timesfm-3.0-pytorch",
    per_core_batch_size=64,
    device="cuda"
)
forecaster = TimesFM3Evaluator(config)

# Simulate a batch of enterprise time series data
batch_size = 4
context_len = 512
horizon = 64

# Multi-channel multivariate target contexts: (num_variates, context_length) per batch item
contexts = [np.random.randn(5, context_len).astype(np.float32) for _ in range(batch_size)]

# Past-only exogenous features (e.g., historical pricing or static flags)
past_only_covariates = [np.random.randn(2, context_len).astype(np.float32) for _ in range(batch_size)]

# Past-and-future known covariates (e.g., calendar features, scheduled promotions across context + horizon)
past_future_covariates = [np.random.randn(3, context_len + horizon).astype(np.float32) for _ in range(batch_size)]

# Execute batched multi-channel inference with full quantile tracking
outputs = list(
    forecaster.predict_batch(
        contexts=contexts,
        horizon=horizon,
        past_only_covariates=past_only_covariates,
        past_future_covariates=past_future_covariates,
        return_quantiles=True,
        use_symmetric_averaging=False,
    )
)

# Inspect output shapes for production validation
for idx, out in enumerate(outputs):
    print(f"Series Group {idx+1} Forecast Shape: {out.forecast.shape}")       # Expects (5, 64)
    print(f"Series Group {idx+1} Quantiles Shape: {out.quantiles.shape}")   # Expects (5, 64, 9)

Series Group 1 Forecast Shape: (5, 64)
Series Group 1 Quantiles Shape: (5, 64, 9)
Series Group 2 Forecast Shape: (5, 64)
Series Group 2 Quantiles Shape: (5, 64, 9)
Series Group 3 Forecast Shape: (5, 64)
Series Group 3 Quantiles Shape: (5, 64, 9)
Series Group 4 Forecast Shape: (5, 64)
Series Group 4 Quantiles Shape: (5, 64, 9)
